# 5) Market coupling — Exercises

**Goals:** compute zonal PTDFs for any GSK, follow the money under nodal and zonal pricing, test whether a better GSK can make flow-based secure, draw a zone around the congestion, and find what a secure zonal market costs.

The setup cell holds the lecture's network, `PTDF`, `real_flows(dispatch)`, the nodal benchmark `NODAL` and `two_zone_market(zone, ntc, flow_based)` — the lecture's market for **any** split of the buses into zones "A" and "B", exchange E from A to B. `NORTH_SOUTH` is the lecture's split (A = South). Power in MW, prices in €/MWh, costs in €/h.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore")
logging.getLogger("linopy").setLevel(logging.ERROR)

# ---- the lecture's network, plants, PTDFs and nodal benchmark ----
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
LINES.index = [f"{a}-{b}" for a, b in zip(LINES.bus0, LINES.bus1)]
X_KM, LIMIT = 0.32, 400.0
LOAD = pd.Series({"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250, "Velebit": 0})[BUSES]
GENS = pd.DataFrame([("wind Velebit", "Velebit", 600, 0.0), ("hydro Split", "Split", 700, 30.0),
                     ("gas Rijeka", "Rijeka", 600, 90.0), ("coal Osijek", "Osijek", 400, 80.0),
                     ("import Zagreb", "Zagreb", 800, 130.0)],
                    columns=["name", "bus", "p_max_MW", "cost_EUR_per_MWh"]).set_index("name")
NORTH_SOUTH = pd.Series({"Zagreb": "B", "Osijek": "B", "Rijeka": "A", "Velebit": "A", "Split": "A"})[BUSES]  # A = South


def ptdf(reference="Zagreb"):
    idx = {b: i for i, b in enumerate(BUSES)}
    b = 1 / (X_KM * LINES.km.to_numpy() / (400 ** 2 / 100))
    A = np.zeros((len(LINES), len(BUSES)))
    A[range(len(LINES)), LINES.bus0.map(idx)] = 1
    A[range(len(LINES)), LINES.bus1.map(idx)] = -1
    B = A.T @ np.diag(b) @ A
    keep = [i for i, name in enumerate(BUSES) if name != reference]
    B_inv = np.zeros_like(B)
    B_inv[np.ix_(keep, keep)] = np.linalg.inv(B[np.ix_(keep, keep)])
    return pd.DataFrame(np.diag(b) @ A @ B_inv, index=LINES.index, columns=BUSES)


PTDF = ptdf()
G = np.zeros((len(BUSES), len(GENS)))
for k, bus in enumerate(GENS.bus):
    G[BUSES.index(bus), k] = 1
GI = pd.Index(GENS.index, name="gen")
COST = xr.DataArray(GENS.cost_EUR_per_MWh.to_numpy(), coords=[GI])


def real_flows(dispatch):
    return pd.Series(PTDF.to_numpy() @ (G @ dispatch[GENS.index].to_numpy() - LOAD.to_numpy()), index=LINES.index)


def nodal():
    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[GI]), name="p")
    m.add_constraints(p.sum() == LOAD.sum(), name="balance")
    M, c0 = PTDF.to_numpy() @ G, PTDF.to_numpy() @ LOAD.to_numpy()
    for l, line in enumerate(LINES.index):
        flow = (p * xr.DataArray(M[l], coords=[GI])).sum() - float(c0[l])
        m.add_constraints(flow <= LIMIT, name=f"upper {line}")
        m.add_constraints(flow >= -LIMIT, name=f"lower {line}")
    m.add_objective((p * COST).sum())
    m.solve(solver_name="highs", output_flag=False)
    lam = float(m.constraints["balance"].dual)
    mu = pd.Series({line: float(m.constraints[f"upper {line}"].dual) + float(m.constraints[f"lower {line}"].dual)
                    for line in LINES.index})
    return {"cost": float(m.objective.value), "dispatch": p.solution.to_pandas(), "lmp": lam + PTDF.T @ mu}


def two_zone_market(zone, ntc=None, flow_based=None):
    """The lecture's market for any split of the buses into zones "A" and "B" (Series bus -> zone).
    ntc limits the exchange E from A to B; flow_based is a list of (line, F_ref, k, E_ref).
    Returns None if the market has no solution."""
    gz = GENS.bus.map(zone)
    zl = LOAD.groupby(zone).sum()
    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[GI]), name="p")
    bound = 1e5 if ntc is None else ntc
    E = m.add_variables(lower=-bound, upper=bound, name="exchange")
    for z, sign in (("A", -1), ("B", 1)):
        gens = list(GENS.index[gz == z])
        produced = sum(p.sel(gen=g) for g in gens) if gens else 0
        m.add_constraints(produced + sign * E == zl.get(z, 0.0), name=f"balance {z}")
    for line, f_ref, k, e_ref in flow_based or []:
        m.add_constraints(k * E <= LIMIT - f_ref + k * e_ref, name=f"fb upper {line}")
        m.add_constraints(k * E >= -LIMIT - f_ref + k * e_ref, name=f"fb lower {line}")
    m.add_objective((p * COST).sum())
    status = m.solve(solver_name="highs", output_flag=False)
    if status[0] != "ok":
        return None
    dispatch = p.solution.to_pandas()
    flows = real_flows(dispatch)
    return {"cost": float(m.objective.value), "dispatch": dispatch, "exchange": float(E.solution), "flows": flows,
            "price": {z: abs(float(m.constraints[f"balance {z}"].dual)) for z in ("A", "B")},
            "secure": bool(flows.abs().max() <= LIMIT + 1e-6)}


NODAL = nodal()

### Warm-ups

**1. Zonal PTDFs for any GSK**

Write `exchange_ptdf(zone, gsk)`: the change in every line's flow per MW exported from zone A to zone B, when each zone's extra output is spread over its buses by the GSK (non-negative weights per bus, normalised within each zone). Check: if each zone's GSK sits on a single bus, the result is the difference of those two buses' PTDF columns; and with the lecture's capacity GSK you get the lecture's values (Zagreb–Rijeka −0.515, Rijeka–Velebit −0.200).

In [2]:
def exchange_ptdf(zone: pd.Series, gsk: pd.Series) -> pd.Series:
    """Flow change on each line (MW) per MW exported from zone A to zone B, under the GSK."""
    ### BEGIN SOLUTION
    g = gsk.reindex(BUSES).fillna(0.0)
    g = g / g.groupby(zone).transform("sum")
    a = PTDF.to_numpy() @ np.where(zone == "A", g, 0.0)
    b = PTDF.to_numpy() @ np.where(zone == "B", g, 0.0)
    return pd.Series(a - b, index=LINES.index)
    ### END SOLUTION

In [3]:
single = exchange_ptdf(NORTH_SOUTH, pd.Series({"Velebit": 1.0, "Osijek": 1.0}))
assert np.allclose(single, PTDF["Velebit"] - PTDF["Osijek"])
capacity = GENS.groupby("bus").p_max_MW.sum()
k = exchange_ptdf(NORTH_SOUTH, capacity)
assert np.isclose(k["Zagreb-Rijeka"], -0.515, atol=1e-3) and np.isclose(k["Rijeka-Velebit"], -0.200, atol=1e-3)
k.round(3)

Zagreb-Rijeka    -0.515
Rijeka-Velebit   -0.200
Velebit-Split     0.116
Zagreb-Split     -0.220
Zagreb-Osijek     0.068
Osijek-Split     -0.265
dtype: float64

**2. Who pays?**

Compare the money flows of the nodal market and the one-zone market with redispatch: what consumers pay at market prices, what generators receive, the congestion rent (nodal) and the redispatch cost (zonal; cost-based, paid by the system operator and recovered from consumers through network tariffs). Write `who_pays()` returning a table with one column per design.

In [4]:
def who_pays() -> pd.DataFrame:
    """Consumer payments, generator revenues, congestion rent and redispatch cost (EUR/h) of two designs."""
    ### BEGIN SOLUTION
    lmp = NODAL["lmp"]
    nodal_pay = float((lmp * LOAD).sum())
    nodal_rev = float(sum(lmp[GENS.bus[g]] * NODAL["dispatch"][g] for g in GENS.index))
    one = two_zone_market(pd.Series("A", index=BUSES))
    price = one["price"]["A"]
    redispatch = float(((NODAL["dispatch"] - one["dispatch"]) * GENS.cost_EUR_per_MWh).sum())
    return pd.DataFrame({
        "nodal": {"consumers pay at market prices": nodal_pay, "generators receive": nodal_rev,
                  "congestion rent (to network owner)": nodal_pay - nodal_rev, "redispatch cost (via tariffs)": 0.0,
                  "consumers pay in total, net of rent": nodal_pay - (nodal_pay - nodal_rev)},
        "one zone + redispatch": {"consumers pay at market prices": price * LOAD.sum(),
                                  "generators receive": price * LOAD.sum(), "congestion rent (to network owner)": 0.0,
                                  "redispatch cost (via tariffs)": redispatch,
                                  "consumers pay in total, net of rent": price * LOAD.sum() + redispatch}})
    ### END SOLUTION

In [5]:
money = who_pays()
assert np.isclose(money.loc["congestion rent (to network owner)", "nodal"], 86077.67, atol=0.1)
assert np.isclose(money.loc["redispatch cost (via tariffs)", "one zone + redispatch"], 10956.3, atol=0.1)
money.round(0)

                                        nodal  one zone + redispatch
consumers pay at market prices       194170.0               171000.0
generators receive                   108092.0               171000.0
congestion rent (to network owner)    86078.0                    0.0
redispatch cost (via tariffs)             0.0                10956.0
consumers pay in total, net of rent  108092.0               181956.0

### Core

**3. Can a better GSK make flow-based secure?**

Run the lecture's flow-based market (North/South, all lines monitored, base case = the one-zone dispatch) with three GSKs for the South: (a) by capacity (the lecture's), (b) only gas at Rijeka and hydro at Split — the plants the market actually adjusts, (c) only wind at Velebit. In the North keep capacity shares. For each, report market cost, exchange and the **real** flow on Rijeka–Velebit, or "no solution". Write `gsk_experiment()`.

In [6]:
def gsk_experiment() -> pd.DataFrame:
    """Flow-based market (all lines) for three South GSKs."""
    ### BEGIN SOLUTION
    base = two_zone_market(pd.Series("A", index=BUSES))
    base_ns = two_zone_market(NORTH_SOUTH)                     # same dispatch, exchange measured South -> North
    north = {"Zagreb": 800.0, "Osijek": 400.0}
    variants = {"(a) capacity": {"Rijeka": 600.0, "Split": 700.0, "Velebit": 600.0},
                "(b) gas and hydro": {"Rijeka": 600.0, "Split": 700.0},
                "(c) wind only": {"Velebit": 1.0}}
    rows = {}
    for label, south in variants.items():
        k = exchange_ptdf(NORTH_SOUTH, pd.Series({**north, **south}))
        fb = [(line, base["flows"][line], k[line], base_ns["exchange"]) for line in LINES.index]
        r = two_zone_market(NORTH_SOUTH, flow_based=fb)
        rows[label] = ({"market_cost": np.nan, "exchange_MW": np.nan, "Rijeka-Velebit_real_MW": np.nan, "result": "no solution"}
                       if r is None else
                       {"market_cost": r["cost"], "exchange_MW": r["exchange"],
                        "Rijeka-Velebit_real_MW": abs(r["flows"]["Rijeka-Velebit"]),
                        "result": "secure" if r["secure"] else "overloaded"})
    return pd.DataFrame(rows).T
    ### END SOLUTION

In [7]:
gsks = gsk_experiment()
assert np.isclose(gsks.loc["(a) capacity", "market_cost"], 104586.0, atol=1)
assert (gsks.result != "secure").all(), "no GSK makes this flow-based market secure"
gsks

                     market_cost exchange_MW Rijeka-Velebit_real_MW  \
(a) capacity       104586.035717  294.139643              446.79677   
(b) gas and hydro            NaN         NaN                    NaN   
(c) wind only        77551.93662  586.201585             526.071141   

                        result  
(a) capacity        overloaded  
(b) gas and hydro  no solution  
(c) wind only       overloaded  

**4. A zone drawn around the congestion**

Draw a zone around Velebit alone (zone A = {Velebit}) and limit its export with an NTC of [600, 500, 485, 400] MW. For each, report market cost, wind output, the real flows on Zagreb–Rijeka and Rijeka–Velebit, and whether the dispatch is secure. Write `velebit_zone()`.

In [8]:
def velebit_zone(ntcs=(600, 500, 485, 400)) -> pd.DataFrame:
    """NTC sweep for a zone around Velebit."""
    ### BEGIN SOLUTION
    zone = pd.Series("B", index=BUSES)
    zone["Velebit"] = "A"
    rows = {}
    for ntc in ntcs:
        r = two_zone_market(zone, ntc=float(ntc))
        rows[ntc] = {"market_cost": r["cost"], "wind_MW": r["dispatch"]["wind Velebit"],
                     "Zagreb-Rijeka_real_MW": abs(r["flows"]["Zagreb-Rijeka"]),
                     "Rijeka-Velebit_real_MW": abs(r["flows"]["Rijeka-Velebit"]),
                     "secure": r["secure"]}
    return pd.DataFrame(rows).T.rename_axis("NTC (MW)")
    ### END SOLUTION

In [9]:
vz = velebit_zone()
assert not vz.secure.any()
assert np.isclose(vz.loc[485, "market_cost"], 81350.0, atol=1)
vz

         market_cost wind_MW Zagreb-Rijeka_real_MW Rijeka-Velebit_real_MW  \
NTC (MW)                                                                    
600          71000.0   600.0            390.945748             490.945748   
500          80000.0   500.0            409.090909             409.090909   
485          81350.0   485.0            411.812683             396.812683   
400          89000.0   400.0             427.23607              327.23607   

         secure  
NTC (MW)         
600       False  
500       False  
485       False  
400       False  

**5. Interpretation**

Why does no GSK make the flow-based market secure — and why is (b), the GSK that describes the market's actual adjustments best, the one with no solution? Why does a zone around Velebit, with an NTC equal to the nodal wind output (485 MW), still overload a line?

<!-- BEGIN SOLUTION -->
No GSK can make this flow-based market secure, because the overload on Rijeka–Velebit is caused by **wind at Velebit**, and the zonal market never curtails it: wind is the South's cheapest plant, so whatever exchange the market chooses, the South cuts gas and hydro first. The GSK only changes what the market **believes** the exchange does to the line. With the capacity GSK (a) it believes 0.200 MW of relief per MW of exchange, cuts the exchange to 294 MW, pays 104,586 €/h and still leaves 447 MW on the line. With the wind-only GSK (c) it believes 0.555 MW of relief per MW — the exchange looks like a strong lever — so it cuts only to 586 MW, pays less (77,552 €/h) and leaves an even larger real overload (526 MW), because the wind it assumed would move never does.

GSK (b) is the honest one: it assumes the South adjusts with gas and hydro, which is exactly what the market does. Under that assumption the exchange barely touches Rijeka–Velebit — 0.035 MW per MW — so bringing the base-case flow of 491 MW down to 400 MW would take an exchange change of about 2,600 MW, more than the whole system's load. The market has **no solution**, and it is right: as long as wind runs at full output, no level of North–South trade relieves the line. Only a decision about **where** in the South power is produced — curtailing wind at Velebit — does, and that is a nodal decision a zonal market cannot take.

The zone drawn around Velebit puts that decision into the market: the NTC now limits exactly the wind export, and at 485 MW Rijeka–Velebit carries 397 MW. But the network has a second congested line. Inside the remaining zone the market chooses by merit order, and it serves 15 MW with gas at Rijeka (90 €/MWh) rather than imports at Zagreb (130 €/MWh) — the only difference from the nodal dispatch. 78.6 % of every MW injected at Rijeka flows over Zagreb–Rijeka, which now carries 412 MW. The nodal solution pays for the dearer imports precisely to keep that line at 400 MW; the zonal market cannot see that a MW at Rijeka and a MW at Zagreb are different.
<!-- END SOLUTION -->

### Challenge

**6. What does a secure zonal market cost?**

For the North/South split and for the zone around Velebit, find the **largest** NTC (searching 0, 5, 10, … 800 MW) at which the zonal market's dispatch is secure, and its market cost. Compare with the nodal cost. Write `secure_ntc()`.

In [10]:
def secure_ntc() -> pd.DataFrame:
    """Largest secure NTC (5 MW steps) and its market cost for two zone splits."""
    ### BEGIN SOLUTION
    velebit = pd.Series("B", index=BUSES)
    velebit["Velebit"] = "A"
    rows = {}
    for label, zone in {"North/South": NORTH_SOUTH, "Velebit / rest": velebit}.items():
        best = None
        for ntc in np.arange(0, 801, 5):
            r = two_zone_market(zone, ntc=float(ntc))
            if r is not None and r["secure"]:
                best = (float(ntc), r["cost"])
        rows[label] = {"largest_secure_NTC_MW": best[0], "market_cost": best[1], "above_nodal": best[1] - NODAL["cost"]}
    return pd.DataFrame(rows).T
    ### END SOLUTION

In [11]:
secure = secure_ntc()
assert (secure.above_nodal > 0).all()
secure.round(1)

                largest_secure_NTC_MW  market_cost  above_nodal
North/South                     155.0     118500.0      36543.7
Velebit / rest                   90.0     121300.0      39343.7

**7. Interpretation**

How much more than the nodal optimum does a secure zonal market cost here, and why is the gap so large? What does that suggest about NTC-based zonal markets in meshed networks with several congested lines?

<!-- BEGIN SOLUTION -->
A secure North/South market needs an NTC of at most 155 MW and costs 118,500 €/h — **36,544 €/h (45 %) more than the nodal optimum**; a secure market with a zone around Velebit needs an NTC of 90 MW and costs 39,344 €/h more. Both NTCs are tiny compared with the 750 MW the zones would like to trade. An NTC has to keep the network secure for whatever the market does inside the zones — and the zonal market's choices inside a zone are exactly the ones that overload lines here: wind at full output behind Rijeka–Velebit, gas at Rijeka rather than imports at Zagreb. With two interacting congested lines, the only way a single number on a single border can rule out every bad dispatch is to choke trade until cheap southern power is barely used, and expensive imports fill the gap.

In a meshed network with several congested lines, an NTC-based zonal market is therefore either insecure — and relies on redispatch afterwards — or very conservative and costly. That is why Europe moved from NTCs to flow-based coupling for its meshed regions, why it still relies on redispatch (lesson 6) for congestion inside zones, and why zone borders are reviewed periodically. Who pays differs too: the money table of task 2 shows that under nodal pricing consumers pay 108,092 €/h net — if all the congestion rent is returned to them — against 181,956 €/h in the one-zone market with redispatch, while society's cost of the secure dispatch is the same 81,956 €/h in both. Pricing is about distribution and signals as much as about cost.
<!-- END SOLUTION -->